# T1 — Inferencia U-Net en Test Set

Evalúa el modelo U-Net (ResNet-34) en el test set completo (18,991 patches) para obtener la tabla comparativa completa de los 3 modelos.

**Modelo:** `unet_resnet34_baseline` — best_checkpoint.pth (época 29, mIoU val=0.4670)

**Artefactos generados:**
- `results/test_evaluation/unet_resnet34_baseline/test_metrics.json`
- `results/test_evaluation/unet_resnet34_baseline/confusion_matrix.npy`
- `results/test_evaluation/unet_resnet34_baseline/iou_per_class.png`
- `results/test_evaluation/unet_resnet34_baseline/confusion_matrix.png`
- `results/test_evaluation/unet_resnet34_baseline/sample_predictions.png`

## 1. Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import json
from datetime import datetime

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True

from scripts.utils import (
    get_device, set_seed,
    create_dataloaders, get_validation_augmentation,
    SegmentationMetrics, N_CLASSES, CLASS_NAMES,
    PATCHES_DIR, PATCHES_INDEX, CHECKPOINTS_DIR, RESULTS_DIR
)

set_seed(42)
device = get_device()

print(f"Device: {device}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. Configuración

In [ ]:
MODEL_NAME = 'unet_resnet34_baseline'
ENCODER    = 'resnet34'
BATCH_SIZE = 8
NUM_WORKERS = 0  # Windows requiere 0

CHECKPOINT_PATH = CHECKPOINTS_DIR / MODEL_NAME / 'best_checkpoint.pth'
OUTPUT_DIR      = RESULTS_DIR / 'test_evaluation' / MODEL_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Modelo:           {MODEL_NAME}")
print(f"Encoder:          {ENCODER}")
print(f"Checkpoint:       {CHECKPOINT_PATH}")
print(f"Output:           {OUTPUT_DIR}")
print(f"Checkpoint existe: {CHECKPOINT_PATH.exists()}")

## 3. Cargar DataLoader de Test

In [ ]:
val_transform = get_validation_augmentation()

dataloaders = create_dataloaders(
    patches_dir=PATCHES_DIR,
    indices_csv=PATCHES_INDEX,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    train_transform=val_transform,
    val_transform=val_transform,
    use_balanced_sampler=False
)

test_loader = dataloaders['test']

print(f"Test set:")
print(f"  Batches:          {len(test_loader)}")
print(f"  Patches totales:  {len(test_loader.dataset):,}")

## 4. Cargar Modelo U-Net

In [ ]:
model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights=None,
    in_channels=3,
    classes=N_CLASSES
)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f"Modelo cargado desde epoch {checkpoint['epoch']}")
print(f"mIoU en validación:  {checkpoint['best_val_miou']:.4f}")
print(f"Parámetros totales:  {sum(p.numel() for p in model.parameters()):,}")

## 5. Inferencia en Test Set

In [ ]:
metrics = SegmentationMetrics(
    num_classes=N_CLASSES,
    ignore_index=0,
    class_names=CLASS_NAMES
)

sample_images = []
sample_masks  = []
sample_preds  = []
N_SAMPLES_SAVE = 20

print(f"Ejecutando inferencia — {len(test_loader)} batches...")

model.eval()
with torch.no_grad():
    for images, masks in tqdm(test_loader, desc="Inferencia U-Net"):
        images = images.to(device)
        masks  = masks.to(device)

        outputs = model(images)
        preds   = torch.argmax(outputs, dim=1)

        metrics.update(preds, masks)

        if len(sample_images) < N_SAMPLES_SAVE:
            sample_images.append(images[0].cpu())
            sample_masks.append(masks[0].cpu())
            sample_preds.append(preds[0].cpu())

print("\nInferencia completada!")

## 6. Métricas finales

In [ ]:
metrics.print_summary()

In [ ]:
summary           = metrics.get_summary(detailed=True)
iou_per_class     = metrics.get_iou_per_class()
dice_per_class    = metrics.get_dice_per_class()
precision_per_class = metrics.get_precision_per_class()
recall_per_class  = metrics.get_class_accuracy()
f1_per_class      = metrics.get_f1_per_class()

print(f"\nmIoU  (sin background): {summary['mIoU']:.4f}")
print(f"Pixel Accuracy:         {summary['pixel_accuracy']:.4f}")
print()
print(f"{'Clase':<22} {'IoU':<8} {'F1':<8} {'Prec':<8} {'Recall':<8}")
print("-" * 56)
for i in range(1, N_CLASSES):
    print(f"{CLASS_NAMES[i]:<22} {iou_per_class[i]:<8.4f} {f1_per_class[i]:<8.4f} "
          f"{precision_per_class[i]:<8.4f} {recall_per_class[i]:<8.4f}")

## 7. Comparativa con DeepLabv3+

In [ ]:
# Cargar métricas del DeepLabv3+ para comparar
deeplab_metrics_path = RESULTS_DIR / 'test_evaluation' / 'deeplabv3_resnet50_baseline' / 'test_metrics.json'

if deeplab_metrics_path.exists():
    with open(deeplab_metrics_path) as f:
        deeplab_results = json.load(f)

    print(f"{'Métrica':<22} {'U-Net':>10} {'DeepLabv3+':>12} {'Δ':>8}")
    print("-" * 54)

    unet_miou   = summary['mIoU']
    deeplab_miou = deeplab_results['metrics']['mIoU']
    print(f"{'mIoU':<22} {unet_miou:>10.4f} {deeplab_miou:>12.4f} {deeplab_miou - unet_miou:>+8.4f}")

    unet_pa   = summary['pixel_accuracy']
    deeplab_pa = deeplab_results['metrics']['pixel_accuracy']
    print(f"{'Pixel Accuracy':<22} {unet_pa:>10.4f} {deeplab_pa:>12.4f} {deeplab_pa - unet_pa:>+8.4f}")

    print()
    print(f"{'Clase':<22} {'U-Net IoU':>10} {'DeepLab IoU':>12} {'Δ':>8}")
    print("-" * 54)
    for i in range(1, N_CLASSES):
        unet_iou   = iou_per_class[i]
        deeplab_iou = deeplab_results['per_class'][CLASS_NAMES[i]]['IoU']
        delta = deeplab_iou - unet_iou
        marker = " ✓" if delta > 0 else ""
        print(f"{CLASS_NAMES[i]:<22} {unet_iou:>10.4f} {deeplab_iou:>12.4f} {delta:>+8.4f}{marker}")
else:
    print("DeepLabv3+ metrics no encontradas — solo se muestran resultados U-Net")

## 8. Guardar resultados

In [ ]:
results = {
    'model_name':       MODEL_NAME,
    'encoder':          ENCODER,
    'checkpoint_epoch': checkpoint['epoch'],
    'val_miou':         float(checkpoint['best_val_miou']),
    'test_date':        datetime.now().isoformat(),
    'test_patches':     len(test_loader.dataset),
    'metrics': {
        'mIoU':            float(summary['mIoU']),
        'mIoU_with_bg':    float(summary['mIoU_with_bg']),
        'pixel_accuracy':  float(summary['pixel_accuracy'])
    },
    'per_class': {}
}

for i in range(N_CLASSES):
    results['per_class'][CLASS_NAMES[i]] = {
        'IoU':       float(iou_per_class[i]),
        'Dice':      float(dice_per_class[i]),
        'Precision': float(precision_per_class[i]),
        'Recall':    float(recall_per_class[i]),
        'F1':        float(f1_per_class[i])
    }

metrics_path = OUTPUT_DIR / 'test_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(results, f, indent=2)

np.save(OUTPUT_DIR / 'confusion_matrix.npy', metrics.confusion_matrix)

print(f"Guardado: {metrics_path}")
print(f"Guardado: {OUTPUT_DIR / 'confusion_matrix.npy'}")

## 9. Visualizaciones

In [ ]:
# --- IoU por clase ---
fig, ax = plt.subplots(figsize=(12, 6))

classes = [CLASS_NAMES[i] for i in range(1, N_CLASSES)]
ious    = [iou_per_class[i] for i in range(1, N_CLASSES)]
colors  = ['#2ecc71' if v > 0.5 else '#f39c12' if v > 0.3 else '#e74c3c' for v in ious]

bars = ax.bar(classes, ious, color=colors, edgecolor='black', linewidth=0.8)
ax.axhline(y=0.5, color='#2ecc71', linestyle='--', alpha=0.7, label='Bueno (>0.5)')
ax.axhline(y=0.3, color='#f39c12', linestyle='--', alpha=0.7, label='Medio (>0.3)')

for bar, iou in zip(bars, ious):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{iou:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('IoU', fontsize=12)
ax.set_xlabel('Clase', fontsize=12)
ax.set_title(f'IoU por Clase en Test Set — {MODEL_NAME}\nmIoU = {summary["mIoU"]:.4f}',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'iou_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Guardado: {OUTPUT_DIR / 'iou_per_class.png'}")

In [ ]:
# --- Matriz de confusión ---
import seaborn as sns

cm = metrics.confusion_matrix.astype('float')
cm_norm = np.nan_to_num(cm / cm.sum(axis=1, keepdims=True))
cm_plot = cm_norm[1:, 1:]  # Sin background
labels  = [CLASS_NAMES[i] for i in range(1, N_CLASSES)]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_plot, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels, yticklabels=labels,
            ax=ax, vmin=0, vmax=1, linewidths=0.5)
ax.set_xlabel('Predicción', fontsize=12)
ax.set_ylabel('Ground Truth', fontsize=12)
ax.set_title(f'Matriz de Confusión Normalizada — {MODEL_NAME}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Guardado: {OUTPUT_DIR / 'confusion_matrix.png'}")

In [ ]:
# --- Comparativa de IoU: U-Net vs DeepLabv3+ ---
if deeplab_metrics_path.exists():
    unet_ious   = [iou_per_class[i] for i in range(1, N_CLASSES)]
    deeplab_ious = [deeplab_results['per_class'][CLASS_NAMES[i]]['IoU'] for i in range(1, N_CLASSES)]

    x = np.arange(len(classes))
    width = 0.35

    fig, ax = plt.subplots(figsize=(13, 6))
    b1 = ax.bar(x - width/2, unet_ious,   width, label=f'U-Net (mIoU={summary["mIoU"]:.4f})',
                color='#3498db', edgecolor='black', linewidth=0.7)
    b2 = ax.bar(x + width/2, deeplab_ious, width, label=f'DeepLabv3+ (mIoU={deeplab_results["metrics"]["mIoU"]:.4f})',
                color='#e67e22', edgecolor='black', linewidth=0.7)

    ax.axhline(y=0.5, color='green', linestyle='--', alpha=0.5)
    ax.set_ylabel('IoU', fontsize=12)
    ax.set_title('Comparativa IoU por Clase — U-Net vs DeepLabv3+ (Test Set)', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=45, ha='right')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=10)

    # Valores encima de barras
    for bar in b1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
    for bar in b2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'comparison_unet_vs_deeplabv3.png', dpi=150, bbox_inches='tight')
    # También guardar en la raíz de test_evaluation para fácil acceso
    comparison_path = RESULTS_DIR / 'test_evaluation' / 'comparison_unet_vs_deeplabv3.png'
    plt.savefig(comparison_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Guardado: {OUTPUT_DIR / 'comparison_unet_vs_deeplabv3.png'}")

In [ ]:
# --- Predicciones de muestra (20 patches) ---
n_show = min(10, len(sample_images))

fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))

for row in range(n_show):
    img_np = sample_images[row].permute(1, 2, 0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f'Imagen #{row}', fontsize=9)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(sample_masks[row].numpy(), cmap='tab10', vmin=0, vmax=8)
    axes[row, 1].set_title('Ground Truth', fontsize=9)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(sample_preds[row].numpy(), cmap='tab10', vmin=0, vmax=8)
    axes[row, 2].set_title('Predicción U-Net', fontsize=9)
    axes[row, 2].axis('off')

    diff = (sample_masks[row] != sample_preds[row]).numpy().astype(float)
    axes[row, 3].imshow(diff, cmap='Reds', vmin=0, vmax=1)
    axes[row, 3].set_title(f'Errores ({diff.mean()*100:.1f}%)', fontsize=9)
    axes[row, 3].axis('off')

plt.suptitle(f'Predicciones de Muestra — {MODEL_NAME}', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Guardado: {OUTPUT_DIR / 'sample_predictions.png'}")

## 10. Resumen final

In [ ]:
print("\n" + "=" * 70)
print("RESUMEN FINAL — U-Net ResNet-34 TEST SET")
print("=" * 70)
print(f"Checkpoint:       Epoch {checkpoint['epoch']}")
print(f"mIoU Validación:  {checkpoint['best_val_miou']:.4f}")
print(f"mIoU Test:        {summary['mIoU']:.4f}")
print(f"Pixel Accuracy:   {summary['pixel_accuracy']:.4f}")
print(f"Diferencia val→test: {summary['mIoU'] - checkpoint['best_val_miou']:+.4f}")
print()
print("Clases fuertes (IoU > 0.5):")
for i in range(1, N_CLASSES):
    if iou_per_class[i] > 0.5:
        print(f"  {CLASS_NAMES[i]:<22}: {iou_per_class[i]:.4f}")
print("Clases débiles (IoU < 0.3):")
for i in range(1, N_CLASSES):
    if iou_per_class[i] < 0.3:
        print(f"  {CLASS_NAMES[i]:<22}: {iou_per_class[i]:.4f}")
print()
print("Archivos generados:")
for f in ['test_metrics.json', 'confusion_matrix.npy', 'iou_per_class.png',
          'confusion_matrix.png', 'comparison_unet_vs_deeplabv3.png', 'sample_predictions.png']:
    path = OUTPUT_DIR / f
    status = "✓" if path.exists() else "✗"
    print(f"  [{status}] {path}")
print("=" * 70)